<a href="https://colab.research.google.com/github/leticiarayane321/rotina_complexos/blob/main/rotina_complexos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Representação e Conversão de Números Complexos**

Números complexos são fundamentais no estudo de sistemas lineares, pois permitem descrever tanto a amplitude quanto a fase de sinais senoidais de forma compacta. Nesta seção, implementamos as duas formas de representação exigidas:

*   **Forma Retangular** ($z = a + bi$): Ideal para operações de soma e subtração, onde
as partes reais e imaginárias são processadas de forma independente
*   **Forma Polar** ($z = r\angle\theta$): Extremamente eficiente para operações de multiplicação e divisão, onde as magnitudes são multiplicadas (ou divididas) e os ângulos são somados (ou subtraídos).


O método convert() em ambas as classes utiliza relações trigonométricas para garantir a transição precisa entre esses dois domínios.

In [1]:
import numpy as np

class Rectangular:
    def __init__(self, real: float, imaginary: float):
        self.real = real
        self.imaginary = imaginary

    def __add__(self, a):
        if isinstance(a, Rectangular):
            return Rectangular(self.real + a.real, self.imaginary + a.imaginary)
        elif isinstance(a, int):
            return Rectangular(self.real + a, self.imaginary)
        elif isinstance(a, float):
            return Rectangular(self.real + a, self.imaginary)
        else:
            raise TypeError('a must be Rectangular, int or float')

    def __sub__(self, a):
        if isinstance(a, Rectangular):
            return Rectangular(self.real - a.real, self.imaginary - a.imaginary)
        elif isinstance(a, int):
            return Rectangular(self.real - a, self.imaginary)
        elif isinstance(a, float):
            return Rectangular(self.real - a, self.imaginary)
        else:
            raise TypeError('a must be Rectangular, int or float')

    def __repr__(self):
        return str(f'{self.real*(abs(self.real) > 1e-5)} + i{self.imaginary*(abs(self.imaginary) > 1e-5)}')

    def convert(self):
        return Polar(np.sqrt((self.real)**2 + (self.imaginary)**2), np.arctan2(self.imaginary, self.real))

class Polar:

    def __init__(self, magnitude : float, angle : float):
        if magnitude < 0:
            raise ValueError('magnitude must be non negative')

        self.magnitude = magnitude
        self.angle = angle

    def __mul__(self, a):
        if isinstance(a, Polar):
            return Polar(self.magnitude * a.magnitude, self.angle + a.angle)
        elif isinstance(a, int):
            return Polar(self.magnitude * a, self.angle)
        elif isinstance(a, float):
            return Polar(self.magnitude * a, self.angle)
        else:
            raise TypeError('a must be Rectangular, int or float')


    def __truediv__(self, a):
        if isinstance(a, Polar):
            return Polar(self.magnitude / a.magnitude, self.angle - a.angle)
        elif isinstance(a, int):
            return Polar(self.magnitude / a, self.angle)
        elif isinstance(a, float):
            return Polar(self.magnitude / a, self.angle)
        else:
            raise TypeError('a must be Rectangular, int or float')

    def convert(self):
        return Rectangular(self.magnitude*(np.cos(self.angle)), self.magnitude*(np.sin(self.angle)))

    def __repr__(self):    
        return str(f'{self.magnitude*(abs(self.magnitude) > 1e-5)}∠{self.angle*(abs(self.angle) > 1e-5)}')
    

In [2]:
False*2

0

**Exemplos Classe Rectangle**

In [3]:
a = Rectangular(np.sqrt(2)/2, np.sqrt(2)/2)
a

0.7071067811865476 + i0.7071067811865476

In [4]:
b = a
a + b

1.4142135623730951 + i1.4142135623730951

In [5]:
a - b

0.0 + i0.0

In [6]:
a.convert()

1.0∠0.7853981633974483

**Exemplos Classe Polar**

In [7]:
a = Polar(1, np.pi/4)
a

1∠0.7853981633974483

In [8]:
b = a
a * b

1∠1.5707963267948966

In [9]:
a/b

1.0∠0.0

In [10]:
a.convert()

0.7071067811865476 + i0.7071067811865475

**Gerenciamento de Abstração: Classe ComplexNumber**

Esta classe funciona como uma interface de alto nível que encapsula as representações Retangular e Polar. Ao instanciar um objeto, o sistema realiza automaticamente a conversão necessária para manter ambas as formas sincronizadas. Isso permite que o usuário realize operações matemáticas utilizando os operadores padrão do Python (+, -, *, /), enquanto a classe decide internamente qual representação é matematicamente mais adequada para o cálculo.

In [11]:
class ComplexNumber:
    def __init__(self, a):
        if isinstance(a, Rectangular):
            self.rectangular = a
            self.polar = a.convert()

        elif isinstance(a, Polar):
            self.polar = a
            self.rectangular = a.convert()

        else:
            raise TypeError('must be Rectangular Polar')

    def __add__(self, a):
        if isinstance(a, ComplexNumber):
            return ComplexNumber(self.rectangular + a.rectangular)
        else:
            return ComplexNumber(self.rectangular + a)


    def __sub__(self, a):
        if isinstance(a, ComplexNumber):
            return ComplexNumber(self.rectangular - a.rectangular)
        else:
            return ComplexNumber(self.rectangular - a)


    def __mul__(self, a):
        if isinstance(a, ComplexNumber):
            return ComplexNumber(self.polar * a.polar)
        else:
            return ComplexNumber(self.polar * a)


    def __truediv__(self, a):
        if isinstance(a, ComplexNumber):
            return ComplexNumber(self.polar/a.polar)
        else:
            return ComplexNumber(self.polar/a)

    def __repr__(self):
        return str(self.rectangular)



In [12]:
z1 = ComplexNumber(Polar(1, np.pi/4))
z1

0.7071067811865476 + i0.7071067811865475

In [13]:
z2 = z1 
z1 + z2

1.4142135623730951 + i1.414213562373095

In [14]:
z1*z2

0.0 + i1.0

In [15]:
(z1*z2).polar

1∠1.5707963267948966

### **Processamento de Sinais Discretos no tempo**

Um sinal discreto é representado como uma sequência (ou array) de amostras, que neste projeto podem ser escalares ou números complexos. A classe Sinal foi desenvolvida para realizar as transformações fundamentais no eixo do tempo e da amplitude:

* **Operações de Amplitude**: Implementação de soma entre sinais.

* **Transformações Temporais**:Deslocamento: Implementa $x[n - k]$, representando atrasos ou avanços temporais.

* C**ontração (Decimação)**: Implementa $x[M \cdot n]$, reduzindo o número de amostras.

* **Expansão (Interpolação)**: Implementa $x[n / L]$ através da inserção de zeros entre as amostras originais, aumentando a taxa de amostragem sem alterar a informação original.


In [16]:

class Sinal:
    def __init__(self, samples, zero = 0):

        if isinstance(samples, list):
            self.samples = samples
            self.zero = zero
        else:
            raise TypeError("Inicialize com uma lista")

    def __add__(self, a):
        if isinstance(a, Sinal):
            return Sinal(self.samples + a.samples)
        raise TypeError("A operação de soma requer outro objeto da classe Sinal")

    def __getitem__(self, i):
        index = self.zero + i
        if index >= 0 and index < len(self.samples):
            return self.samples[index]
        else:
            return 0

    def __mul__(self, h):
        if not isinstance(h, Sinal):
            raise TypeError("Para convolução, o argumento deve ser um objeto Sinal")

        u = self.samples
        h_samples = h.samples
        N = len(u)
        M = len(h_samples)

        tamanho_final = N + M - 1
        resultado_amostras = []

        for k in range(tamanho_final):
            soma_k = ComplexNumber(Rectangular(0, 0))

            "O loop interno aplica o somatório Σ de n=0 até k"
            for n in range(k + 1):
                if n < N and (k - n) < M:

                    termo = u[n] * h_samples[k - n]
                    soma_k = soma_k + termo

            resultado_amostras.append(soma_k)

        return Sinal(resultado_amostras)

    def deslocamento(self, k):
        " Para x[n - k] "
        return Sinal(self.samples, zero = self.zero + k)

    def contracao(self, fator):
        " Para x[fator * n] (Decimação) "
        return Sinal(self.samples[::fator])

    def expansao(self, fator):
        " Para x[n / fator] (Interpolação com zero) "
        novo_tamanho = len(self.samples) * fator

        aux = [ComplexNumber(Rectangular(0, 0))] * novo_tamanho

        aux[::fator] = self.samples
        return Sinal(aux)

    def __repr__(self):
        return f"{self.samples}"


In [17]:
for i in range(-1, 0):
    print(i)

-1


In [18]:
z1 = ComplexNumber(Polar(1, np.pi/4))
samples = [z1]
for i in range(1, 8):
    samples.append(samples[i-1]*z1)
u = Sinal(samples)
u

[0.7071067811865476 + i0.7071067811865475, 0.0 + i1.0, -0.7071067811865475 + i0.7071067811865476, -1.0 + i0.0, -0.7071067811865477 + i-0.7071067811865475, -0.0 + i-1.0, 0.7071067811865474 + i-0.7071067811865477, 1.0 + i-0.0]

In [19]:
u[0].polar

1∠0.7853981633974483

**Deslocamento**

In [20]:
u_deslocado = u.deslocamento(1)
u_deslocado

[0.7071067811865476 + i0.7071067811865475, 0.0 + i1.0, -0.7071067811865475 + i0.7071067811865476, -1.0 + i0.0, -0.7071067811865477 + i-0.7071067811865475, -0.0 + i-1.0, 0.7071067811865474 + i-0.7071067811865477, 1.0 + i-0.0]

In [21]:
u_deslocado = u.deslocamento(3)
u_deslocado

[0.7071067811865476 + i0.7071067811865475, 0.0 + i1.0, -0.7071067811865475 + i0.7071067811865476, -1.0 + i0.0, -0.7071067811865477 + i-0.7071067811865475, -0.0 + i-1.0, 0.7071067811865474 + i-0.7071067811865477, 1.0 + i-0.0]

In [22]:
u_deslocado = u.deslocamento(-2)
u_deslocado

[0.7071067811865476 + i0.7071067811865475, 0.0 + i1.0, -0.7071067811865475 + i0.7071067811865476, -1.0 + i0.0, -0.7071067811865477 + i-0.7071067811865475, -0.0 + i-1.0, 0.7071067811865474 + i-0.7071067811865477, 1.0 + i-0.0]

**Contração e Expansão**

In [23]:
h = Sinal([1, 2, 3, 4, 5, 6])
h

[1, 2, 3, 4, 5, 6]

In [24]:
h_exp = h.expansao(2)
h_exp

[1, 0 + i0, 2, 0 + i0, 3, 0 + i0, 4, 0 + i0, 5, 0 + i0, 6, 0 + i0]

In [25]:
h_contr = h.contracao(2)
h_contr

[1, 3, 5]

In [26]:
h_exp_contr = h_exp.contracao(2)
h_exp_contr

[1, 2, 3, 4, 5, 6]

### **Convolução Linear de Sinais Finitos**:

 A convolução é a operação matemática base para determinar a saída de um sistema Linear e Invariante no Tempo (LIT). Seguindo as orientações de sala de aula, implementamos a convolução, usando o método "__mul__" , de dois sinais finitos $u[n]$ e $h[n]$ através da fórmula do somatório:

$$(u * h)[k] = \sum_{n=0}^{k} u[n] \cdot h[k - n]$$


In [27]:
u = Sinal([1, 2, 3, 4, 5, 6])
h = Sinal([1])
u*h

[1 + i0, 2 + i0, 3 + i0, 4 + i0, 5 + i0, 6 + i0]

In [28]:
h = Sinal([0, 1])
u*h

[0 + i0, 1 + i0, 2 + i0, 3 + i0, 4 + i0, 5 + i0, 6 + i0]

In [29]:
h = Sinal([5, 4, 3, 2, 1])
u*h

[5 + i0, 14 + i0, 26 + i0, 40 + i0, 55 + i0, 70 + i0, 50 + i0, 32 + i0, 17 + i0, 6 + i0]

In [30]:
h*u

[5 + i0, 14 + i0, 26 + i0, 40 + i0, 55 + i0, 70 + i0, 50 + i0, 32 + i0, 17 + i0, 6 + i0]